In [8]:
%pip install tree-sitter tree_sitter_languages -q

Note: you may need to restart the kernel to use updated packages.


In [7]:
Language('python')

TypeError: Language.__init__() missing 1 required positional argument: 'name'

In [6]:
from tree_sitter import Language, Parser

In [9]:
from tree_sitter_languages import get_language, get_parser

In [10]:
language = get_language('python')
parser = get_parser('python')

/Users/kylekelley/Library/Caches/pypoetry/virtualenvs/chatlab-0EWIBOuo-py3.11/lib/python3.11/site-packages/tree_sitter/__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


In [5]:
import tree_sitter_python as tspython
from tree_sitter import Language, Parser

PY_LANGUAGE = Language(tspython.language(), "python")

ModuleNotFoundError: No module named 'tree_sitter_python'

In [ ]:
import tree_sitter
import openai

# Load the tree-sitter language and parser
LANGUAGE = tree_sitter.Language('path/to/language.so', 'language_name')
parser = tree_sitter.Parser()
parser.set_language(LANGUAGE)

def execute_query(code, query):
    # Parse the code using tree-sitter
    tree = parser.parse(bytes(code, 'utf8'))
    
    # Execute the tree-sitter query on the parsed tree
    query = LANGUAGE.query(query)
    captures = query.captures(tree.root_node)
    
    # Extract the relevant information from the captures
    results = []
    for capture in captures:
        node = capture[0]
        start_pos = node.start_point
        end_pos = node.end_point
        text = code[start_pos[0]:end_pos[0]]
        results.append({
            'start_line': start_pos[0],
            'start_column': start_pos[1],
            'end_line': end_pos[0],
            'end_column': end_pos[1],
            'text': text
        })
    
    return results

def generate_response(query_results, prompt):
    # Prepare the input for the ChatGPT API
    input_text = f"Query Results:\n{query_results}\n\nPrompt: {prompt}"
    
    # Call the ChatGPT API to generate a response
    response = openai.Completion.create(
        engine='davinci',
        prompt=input_text,
        max_tokens=100,
        n=1,
        stop=None,
        temperature=0.7
    )
    
    # Extract the generated response
    generated_text = response.choices[0].text.strip()
    
    return generated_text

# Example usage
code = '''
def hello_world():
    print("Hello, World!")
'''

query = '''
(function_definition
  name: (identifier) @function_name
  body: (block
    (expression_statement
      (call
        function: (identifier) @call_name
        arguments: (argument_list) @arguments
      )
    )
  )
)
'''

query_results = execute_query(code, query)
prompt = "Explain the purpose of the function and what it does."
response = generate_response(query_results, prompt)

print(response)